# Batch OCR Processing with Umi-OCR

This notebook demonstrates how to process multiple images in batch using Umi-OCR's HTTP API.

## Prerequisites

1. Umi-OCR must be running with HTTP service enabled
2. Have multiple images ready for batch processing

In [ ]:
# Install required packages
!pip install requests pillow pandas tqdm

In [ ]:
import requests
import base64
import json
from pathlib import Path
from PIL import Image
import pandas as pd
from tqdm.notebook import tqdm
import time

## Configuration

In [ ]:
# Umi-OCR configuration
UMI_OCR_HOST = "127.0.0.1"
UMI_OCR_PORT = 1224
BASE_URL = f"http://{UMI_OCR_HOST}:{UMI_OCR_PORT}"

# Batch processing configuration
IMAGE_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp']
DELAY_BETWEEN_REQUESTS = 0.1  # seconds

## Helper Functions

In [ ]:
def image_to_base64(image_path):
    """Convert an image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def ocr_image(image_base64, options=None):
    """Perform OCR on a base64 encoded image."""
    url = f"{BASE_URL}/api/ocr"
    payload = {"base64": image_base64}
    if options:
        payload.update(options)
    
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        return {'code': -1, 'data': str(e)}

def extract_text_from_result(result):
    """Extract text from OCR result."""
    if result.get('code') == 100:
        texts = [item.get('text', '') for item in result.get('data', [])]
        return '\n'.join(texts)
    return ''

## 1. Find Images in Directory

Recursively search for all images in a directory:

In [ ]:
def find_images(directory, recursive=True):
    """Find all image files in a directory.
    
    Args:
        directory: Path to search for images
        recursive: Whether to search subdirectories
        
    Returns:
        List of Path objects for found images
    """
    directory = Path(directory)
    images = []
    
    if recursive:
        for ext in IMAGE_EXTENSIONS:
            images.extend(directory.rglob(f"*{ext}"))
            images.extend(directory.rglob(f"*{ext.upper()}"))
    else:
        for ext in IMAGE_EXTENSIONS:
            images.extend(directory.glob(f"*{ext}"))
            images.extend(directory.glob(f"*{ext.upper()}"))
    
    return sorted(set(images))

# Example: Find images in a directory
image_directory = "./images"  # Change this to your images directory

if Path(image_directory).exists():
    image_files = find_images(image_directory)
    print(f"Found {len(image_files)} images")
    for img in image_files[:5]:  # Show first 5
        print(f"  - {img}")
    if len(image_files) > 5:
        print(f"  ... and {len(image_files) - 5} more")
else:
    print(f"Directory '{image_directory}' not found. Please create it and add some images.")

## 2. Batch OCR Processing

Process multiple images with progress tracking:

In [ ]:
def batch_ocr(image_paths, options=None, delay=0.1):
    """Process multiple images with OCR.
    
    Args:
        image_paths: List of image file paths
        options: OCR options dictionary
        delay: Delay between requests (seconds)
        
    Returns:
        List of dictionaries containing results
    """
    results = []
    
    for image_path in tqdm(image_paths, desc="Processing images"):
        try:
            # Convert to base64
            image_b64 = image_to_base64(image_path)
            
            # Perform OCR
            result = ocr_image(image_b64, options)
            
            # Store result
            results.append({
                'path': str(image_path),
                'filename': image_path.name,
                'success': result.get('code') == 100,
                'text': extract_text_from_result(result),
                'code': result.get('code'),
                'raw_data': result.get('data', [])
            })
            
            # Delay between requests to avoid overwhelming the server
            time.sleep(delay)
            
        except Exception as e:
            results.append({
                'path': str(image_path),
                'filename': image_path.name,
                'success': False,
                'text': '',
                'code': -1,
                'error': str(e)
            })
    
    return results

# Process images if directory exists
if Path(image_directory).exists() and len(image_files) > 0:
    print(f"Processing {len(image_files)} images...")
    batch_results = batch_ocr(image_files, delay=DELAY_BETWEEN_REQUESTS)
    
    # Summary
    successful = sum(1 for r in batch_results if r['success'])
    print(f"\nProcessing complete!")
    print(f"Successful: {successful}/{len(batch_results)}")
else:
    print("No images to process. Please add images to the directory.")
    batch_results = []

## 3. View Results in DataFrame

In [ ]:
if batch_results:
    # Create DataFrame
    df = pd.DataFrame([{
        'Filename': r['filename'],
        'Success': r['success'],
        'Text Preview': r['text'][:100] + '...' if len(r['text']) > 100 else r['text'],
        'Text Length': len(r['text']),
        'Code': r['code']
    } for r in batch_results])
    
    display(df)
else:
    print("No results to display.")

## 4. Export Results

Save the results to various formats:

In [ ]:
def export_results(results, output_dir="./output"):
    """Export OCR results to multiple formats.
    
    Args:
        results: List of result dictionaries
        output_dir: Directory to save output files
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    # 1. Export as JSON
    json_path = output_path / "ocr_results.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved JSON: {json_path}")
    
    # 2. Export as CSV
    csv_path = output_path / "ocr_results.csv"
    df = pd.DataFrame([{
        'filename': r['filename'],
        'path': r['path'],
        'success': r['success'],
        'text': r['text'],
        'code': r['code']
    } for r in results])
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"Saved CSV: {csv_path}")
    
    # 3. Export individual text files
    text_dir = output_path / "text_files"
    text_dir.mkdir(exist_ok=True)
    for result in results:
        if result['success'] and result['text']:
            filename = Path(result['filename']).stem + '.txt'
            text_path = text_dir / filename
            with open(text_path, 'w', encoding='utf-8') as f:
                f.write(result['text'])
    print(f"Saved text files: {text_dir}")
    
    # 4. Export summary report
    report_path = output_path / "summary_report.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("OCR Batch Processing Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Total images processed: {len(results)}\n")
        successful = sum(1 for r in results if r['success'])
        f.write(f"Successful: {successful}\n")
        f.write(f"Failed: {len(results) - successful}\n\n")
        
        f.write("Individual Results:\n")
        f.write("-" * 50 + "\n")
        for result in results:
            f.write(f"\nFile: {result['filename']}\n")
            f.write(f"Status: {'Success' if result['success'] else 'Failed'}\n")
            if result['success']:
                f.write(f"Text length: {len(result['text'])} characters\n")
    print(f"Saved report: {report_path}")

if batch_results:
    export_results(batch_results)
else:
    print("No results to export.")

## 5. Advanced: Parallel Processing

For large batches, you can process images in parallel (use with caution):

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_image(image_path, options=None):
    """Process a single image and return result."""
    try:
        image_b64 = image_to_base64(image_path)
        result = ocr_image(image_b64, options)
        return {
            'path': str(image_path),
            'filename': image_path.name,
            'success': result.get('code') == 100,
            'text': extract_text_from_result(result),
            'code': result.get('code'),
            'raw_data': result.get('data', [])
        }
    except Exception as e:
        return {
            'path': str(image_path),
            'filename': image_path.name,
            'success': False,
            'text': '',
            'code': -1,
            'error': str(e)
        }

def batch_ocr_parallel(image_paths, options=None, max_workers=2):
    """Process images in parallel with limited workers.
    
    Note: Use max_workers=2 or 3 to avoid overwhelming Umi-OCR.
    """
    results = []
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_path = {executor.submit(process_single_image, path, options): path 
                          for path in image_paths}
        
        # Process completed tasks
        for future in tqdm(as_completed(future_to_path), total=len(image_paths), 
                          desc="Processing (parallel)"):
            result = future.result()
            results.append(result)
    
    return results

# Example (commented out to avoid accidental parallel execution)
# if Path(image_directory).exists() and len(image_files) > 0:
#     print("Processing images in parallel...")
#     parallel_results = batch_ocr_parallel(image_files, max_workers=2)
#     print(f"Parallel processing complete! Processed {len(parallel_results)} images")

print("Parallel processing example is commented out.")
print("Uncomment the code above to test parallel processing.")

## Summary

This notebook demonstrated:
- Finding images in directories
- Batch OCR processing with progress tracking
- Viewing results in pandas DataFrame
- Exporting results to multiple formats (JSON, CSV, TXT)
- Parallel processing for large batches

## Tips for Batch Processing

1. **Processing Speed**: Add appropriate delays between requests to avoid overwhelming Umi-OCR
2. **Error Handling**: Always check the response code and handle failures gracefully
3. **Memory Usage**: For very large batches, consider processing in chunks
4. **Parallel Processing**: Use with caution (max 2-3 workers) as Umi-OCR may not handle high concurrency well

## Next Steps

- See `03_qrcode_operations.ipynb` for QR code examples
- Refer to [HTTP API Documentation](../../docs/http/README.md) for more options